# Instrução de Estudo de Normalização

O objetivo deste procedimento é padronizar a coluna de "Assuntos" da base da PGE, transformando diversas variações de texto em termos canônicos (padronizados).

### 1. Preparação do Ambiente
Antes de começar, certifique-se de que o ambiente está configurado conforme o `README.md`:
* **Local:** Crie o notebook em `notebooks/estudo-similarididade/normalizacao_assuntos.ipynb`.
* **Dependências:** Adicione a biblioteca de similaridade caso não esteja instalada:
    ```bash
    uv add thefuzz
    ```

### 2. Fase de Pré-processamento (Limpeza de Strings)
Carregue os dados cedidos pelo Beto e aplique a normalização básica para reduzir o ruído antes do cálculo de similaridade:
1.  **Extração:** Converta a coluna de assuntos em uma **Lista de Strings únicas**.
2.  **Sanitização:** Para cada string da lista, aplique:
    * `lower()`: Converter tudo para minúsculo.
    * `strip()`: Remover espaços em branco no início e fim.
    * **Remover Acentos:** Utilize a biblioteca `unicodedata` ou `unidecode`.
    * **Regex:** Remova caracteres especiais excessivos (pontuação desnecessária).

### 3. Matriz de Similaridade e Fuzzy Matching
Utilizaremos a biblioteca `thefuzz` para agrupar termos escritos de forma diferente, mas que se referem ao mesmo assunto.

1.  **Cálculo:** Utilize `fuzz.ratio` ou `fuzz.token_sort_ratio` para comparar as strings.
2.  **Threshold (Limiar):** Defina um valor de corte (ex: 85 ou 90). 
    * Se `score >= threshold`, os termos são candidatos a serem o mesmo assunto.
3.  **Matriz:** O resultado deve ser uma estrutura que identifique quais termos são "irmãos" baseados na similaridade.



### 4. Construção do Dicionário "Verdade" (Mapping)
Esta é a fase mais importante. Você deve criar um mapeamento onde a **Chave (Key)** é o nome correto/padrão e o **Valor (Value)** é a lista de variações encontradas.

* **Exemplo de Estrutura:**
    ```python
    dicionario_verdade = {
        "FORNECIMENTO DE MEDICAMENTOS": [
            "fornecimento de medicamento",
            "medica mentos",
            "fornecimento de remedio"
        ],
        "TRATAMENTO MEDICO-HOSPITALAR": [
            "tratamento medico hospitalar",
            "trat. medico"
        ]
    }
    ```
* **Iteração:** Se um termo não se encaixar em nenhum grupo com o threshold definido, ele deve ser marcado para **Revisão Manual (Outliers)**.

### 5. Exportação e Segurança de Dados
Após validar a normalização:
1.  **Gerar o Arquivo:** Salve o resultado final como `data/PGE.GPDR_normalizado.json`.
2.  **Regra de Ouro do Git:** > **Atenção:** O arquivo `PGE.GPDR_normalizado.json` **NÃO** deve ser enviado para o GitHub. Ele já está listado no `.gitignore`. O repositório deve conter apenas o código (Notebook) e a lógica de processamento.
3.  **Entrega:** Envie o arquivo `.json` gerado diretamente para o Étore e o Beto (via E-mail) para validação da qualidade dos dados.

---

### Dicas para o Pesquisador:
* **Modularize:** Tente criar funções para a limpeza (passo 2) para que o código fique organizado.
* **Documente:** Use as células de Markdown do Notebook para explicar por que escolheu determinado *threshold* de similaridade.
* **Performance:** Se a lista de assuntos for muito grande (ex: > 10.000 itens únicos), o cálculo da matriz de similaridade pode demorar. Se precisar de ajuda com performance, fale com o Étore.

# Estudo Similaridade

## 1. Preparação do Ambiente

In [ ]:
import json
import re
from collections import defaultdict
import pandas as pdm
from thefuzz import fuzz
import unidecode

## 2. Fase de Pré-processamento (Limpeza de Strings)

In [ ]:
def limpar_texto(texto: str) -> str:
    
    texto = texto.lower().strip()
    texto = unidecode.unidecode(texto)
    texto = re.sub(r"\s+", " ", texto)
    texto = re.sub(r"[^\w\s\-]", "", texto)

    return texto.strip()

with open("../../data/PGE.GPDR.json", "r", encoding="utf-8") as f:
    dados = json.load(f)

    df = pd.DataFrame(dados)

df["Assuntos"] = df["Assuntos"].apply(limpar_texto)

## 3. Matriz de Similaridade e Fuzzy Matching

In [ ]:
assuntos_individuais = []

for assunto in df["Assuntos"]:

    partes = re.split(r" - | / ", assunto)

    for parte in partes:

        parte = parte.strip()

        if parte != "":

            assuntos_individuais.append(parte)

assuntos_unicos = list(set(assuntos_individuais))


###  Foi utilizado o valor 95 para focar apenas as pequenas variações como diferenças de plural, hífen e espaçamento.

In [ ]:
threshold = 95

for assunto in assuntos_unicos:

    for outro_assunto in assuntos_unicos:

        if assunto == outro_assunto:
            continue

        if re.search(r"\d", assunto) or re.search(r"\d", outro_assunto):
            continue

        if (
    ("recalculo" in assunto and "calculo" in outro_assunto)
    or
    ("calculo" in assunto and "recalculo" in outro_assunto)
           ):
            continue

        score = fuzz.ratio(assunto, outro_assunto)

        if score >= threshold:

            print("ASSUNTO 1:", assunto)
            print("ASSUNTO 2:", outro_assunto)
            print("SIMILARIDADE:", score)
            print()

## 4. Construção do Dicionário "Verdade" (Mapping)

In [ ]:
dicionario_verdade = {

    "inclusao abono ou piso salarial": [
        "inclusao abono ou piso salarial",
        "inclusao abono e piso salarial"
    ],

    "adicional de desempenho da saude": [
        "adicional de desempenho da saude",
        "adicional desempenho da saude"
    ],

    "adicionais gratificacoes": [
        "adicionaisgratificacoes",
        "adicionais  gratificacoes"
    ],

    "processo administrativo disciplinar ou sindicancia": [
        "processo administrativo disciplinar  sindicancia",
        "processo administrativo disciplinar ou sindicancia"
    ],

    "atribuicao servidores estatutarios": [
        "atribuicao  servidores estatutarios",
        "atribuicao  servidor estatutarios"
    ],

    "servidores estatutarios": [
        "servidor estatutarios",
        "servidores estatutarios"
    ],

    "licenca-premio em pecunia": [
        "licenca premio em pecunia",
        "licenca-premio em pecunia"
    ],

    "quinquenios": [
        "quinquenios",
        "quinquenio"
    ],

    "sexta-parte": [
        "sexta-parte",
        "sexta parte"
    ],

    "adicionais por tempo de servico sobre os vencimentos integrais": [
        "adicionais por tempo de servico sobre vencimentos integrais",
        "adicionais por tempo de servico sobre os vencimentos integrais"
    ]
}

## 5. Exportação e Segurança de Dados

In [ ]:
for i, assunto in df["Assuntos"].items():

    for canonico, variacoes in dicionario_verdade.items():

        for variacao in variacoes:

            assunto = assunto.replace(variacao, canonico)

    df.at[i, "Assuntos"] = assunto

resultado = df.to_dict(orient="records")

with open("../../data/PGE.GPDR_normalizado.json", "w", encoding="utf-8") as f:

    json.dump(
        resultado,
        f,
        ensure_ascii=False,
        indent=4
    )